# Root Node Strategy Visualization

Compare how different root node strategies select the initial polygons for the relaxed solution.

In [ ]:
import sys

from tspn_bnb2.misc import FULLWIDEFIGURE

sys.path.insert(0, "../../examples")

import matplotlib.pyplot as plt
import numpy as np
from _utils import generate_random_instance
from shapely import Polygon as ShapelyPolygon
from shapely.plotting import plot_line, plot_polygon
from tspn_bnb2 import solve_annotated_instance
from tspn_bnb2.core import TourElement, compute_tour_from_sequence
from tspn_bnb2.operations import convert_to_tspn_instance, to_shapely_linestring
from tspn_bnb2.order_annotation import add_order_annotations
from tspn_bnb2.schemas import AnnotatedInstance

In [ ]:
# Original instance (no triangles) with order annotations
rng = np.random.default_rng(1337)
base_instance = generate_random_instance(num_polygons=50, seed=42)
base_instance.polygons = [
    base_instance.polygons[i]
    for i in rng.choice(
        len(base_instance.polygons), size=len(base_instance.polygons), replace=False
    )
]

In [ ]:
# Compute bbox of the original instance
from shapely.ops import unary_union

all_polys_union = unary_union(base_instance.polygons)
minx, miny, maxx, maxy = all_polys_union.bounds
bbox_w, bbox_h = maxx - minx, maxy - miny
cx_inst, cy_inst = (minx + maxx) / 2, (miny + maxy) / 2

# Add 4 big triangles pointing inward, sized ~bbox/2
tri_size = max(bbox_w, bbox_h) / 1.6
tri_radius = max(bbox_w, bbox_h) * 0.7  # place outside the instance
tri_width = tri_size * 0.1
triangles = []
for i in range(4):
    angle = 2 * np.pi * i / 4 + np.pi / 4  # offset by 45 degrees
    # Triangle base center, placed outside the bbox
    bx = cx_inst + tri_radius * np.cos(angle)
    by = cy_inst + tri_radius * np.sin(angle)
    # Direction toward instance center
    dx, dy = -np.cos(angle), -np.sin(angle)
    # Perpendicular direction
    px, py = -dy, dx
    tip = (bx + tri_size * dx, by + tri_size * dy)
    base1 = (bx + tri_width * px, by + tri_width * py)
    base2 = (bx - tri_width * px, by - tri_width * py)
    triangles.append(ShapelyPolygon([base1, base2, tip, base1]))

In [ ]:
original_instance = add_order_annotations(AnnotatedInstance(polygons=base_instance.polygons))
original_tspn = convert_to_tspn_instance(original_instance)

triangle_instance = AnnotatedInstance(polygons=base_instance.polygons + triangles)
triangle_instance = add_order_annotations(triangle_instance)
triangle_tspn = convert_to_tspn_instance(triangle_instance)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 4))
for poly in triangle_instance.polygons:
    face_rgba = (0.8, 0.8, 0.8, 0.3)
    plot_polygon(poly, ax=ax, facecolor=face_rgba, edgecolor="black", add_points=False)
    ax.set_aspect("equal")
    ax.set_axis_off()

In [ ]:
def capture_root_trajectory(inst, root_strategy):
    """Run solver and capture the root node's relaxed trajectory."""
    captured = {}

    def callback(context, _cap=captured):
        if context.num_iterations == 1:
            relaxed = context.get_relaxed_solution()
            _cap["trajectory"] = to_shapely_linestring(relaxed.get_trajectory())

    solve_annotated_instance(inst, timelimit=5, root=root_strategy, callback=callback)
    return captured.get("trajectory")


def manual_root_trajectory(tspn_inst, indices):
    """Compute a relaxed trajectory through manually chosen polygon indices."""
    elements = [TourElement(tspn_inst, idx) for idx in indices]
    traj = compute_tour_from_sequence(elements, False, False)
    return to_shapely_linestring(traj)


# Column 1: Random on original instance as we can not seed a seed via our interface.
random_indices = sorted(rng.choice(len(original_instance.polygons), size=3, replace=False))
random_traj = manual_root_trajectory(original_tspn, random_indices)

# Column 2: LongestTriple on original instance
longest_triple_traj = capture_root_trajectory(original_instance, "LongestTriple")

# Column 3: OrderRoot on original instance
order_root_orig_traj = capture_root_trajectory(original_instance, "OrderRoot")

# Column 4: OrderRoot on instance with triangles
order_root_tri_traj = capture_root_trajectory(triangle_instance, "OrderRoot")

strategies = [
    ("Random", original_instance, random_traj),
    ("LongestTriple", original_instance, longest_triple_traj),
    ("OrderRoot", original_instance, order_root_orig_traj),
    ("OrderRoot (triangles)", triangle_instance, order_root_tri_traj),
]

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=FULLWIDEFIGURE)
cmap = plt.get_cmap("tab20")

for ax, (_, inst, traj) in zip(axs, strategies):
    for j, poly in enumerate(inst.polygons):
        if traj is not None and traj.distance(poly) < 0.001:
            color = cmap(j % cmap.N)
            face_rgba = (*color[:3], 0.5)
        else:
            face_rgba = (0.7, 0.7, 0.7, 0.5)
        plot_polygon(
            poly, ax=ax, facecolor=face_rgba, edgecolor="black", add_points=False, linewidth=0.5
        )
    if traj is not None:
        # plot vertices manually
        plot_line(traj, ax=ax, add_points=False, color="black", linewidth=1, zorder=10)
        x, y = traj.xy
        ax.plot(x, y, "o", markersize=3, color="black", zorder=10)

    ax.set_aspect("equal")
    ax.set_axis_off()

fig.tight_layout()
plt.savefig("out/root_node_strategies.pdf", bbox_inches="tight", facecolor="white")
plt.show()